# Phase 1 — Budget-matched Random Search (EXPERIMENTS)

Runs the experiment and saves raw results. **Does not** produce tables or figures —
that is `analysis.ipynb`, so you never have to re-run this to change a plot.

**Pre-registered decisions (locked before any result was seen — do not change):**

1. **Budget is measured, not assumed.** The nominal 40 + 40x20 = 840 is an upper bound;
   DEAP re-evaluates only individuals whose fitness was invalidated. Measured on seed 0:
   Breast Cancer 745, Colon 840, Leukemia 840. This notebook re-measures across several
   seeds and uses the mean rounded up, per dataset.
2. **Random Search returns a FRONT, not one subset.** The non-dominated set of its samples
   is extracted w.r.t. the same two objectives, then the same two reductions are applied
   (front-union, knee) using the existing functions in `src/stability.py`.
3. **Cardinality:** `k ~ Uniform{1..min(p,50)}`, then k features uniformly without
   replacement. Sampling from NSGA-II's initialisation distribution is forbidden — on
   Leukemia that is ~Poisson(20), which would essentially never reach the 2-6 feature
   region where NSGA-II's knee sits, making the baseline artificially weak.
   K_max = 50 covers every cardinality observed in NSGA-II fronts (max: Colon 19,
   Leukemia 15, Breast Cancer 7).
4. **Chance baselines are recomputed per method** from that method's own subset sizes.
5. **All comparisons are descriptive.** No significance test: M = 10, the 45 pairs are not
   independent, and both interval estimators for Phi were previously shown unreliable here.

**Runtime warning:** roughly 1.5-2 hours total. Results are checkpointed after each
(dataset, condition); re-running skips completed work.


## 1. Environment, versions, seeding policy

In [ ]:
import sys, platform
print("Python:", sys.version.split()[0], "|", platform.system(), platform.release())

# If anything below is missing, install with:
#   pip install numpy scipy scikit-learn deap
# No internet is required at run time; the .mat files must already be in data/.

import numpy, scipy, sklearn, deap
for m in (numpy, scipy, sklearn, deap):
    print(f"{m.__name__:>12}: {m.__version__}")

# SEEDING POLICY
#   bootstrap draws : np.random.default_rng(0), drawn 10x in sequence -> reproduces
#                     exactly the draws used by the saved NSGA-II runs
#   NSGA-II seeds   : 0-9 (data-resampling), 100-109 (fixed-data)
#   Random Search   : seeds 1000-1009 (data-resampling), 2000-2009 (fixed-data)
#                     kept disjoint from NSGA-II seeds so the two methods cannot
#                     accidentally share a random stream

## 2. CONFIG — the only place paths or parameters are set

In [ ]:
from pathlib import Path

# PROJ_ROOT should contain: src/, data/, results/
PROJ_ROOT = Path.cwd()
if not (PROJ_ROOT / "src").exists() and (PROJ_ROOT.parent / "src").exists():
    PROJ_ROOT = PROJ_ROOT.parent

SRC_DIR     = PROJ_ROOT / "src"
DATA_DIR    = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

OUT_FILE       = RESULTS_DIR / "phase1_random_search.json"
BUDGET_FILE    = RESULTS_DIR / "phase1_measured_budget.json"

DATASETS       = ["breast_cancer", "colon", "leukemia"]
N_RUNS         = 10          # matches NSGA-II
K_MAX_CAP      = 50          # decision 3
BUDGET_SEEDS   = [0, 1, 2]   # seeds used to MEASURE the NSGA-II budget
RS_SEED_BASE   = {"bootstrap": 1000, "fixed": 2000}

# NSGA-II config, repeated here ONLY so the budget measurement matches the saved runs
POP_SIZE, N_GEN = 40, 20

assert SRC_DIR.exists(), f"src/ not found under {PROJ_ROOT}"
assert DATA_DIR.exists(), f"data/ not found under {PROJ_ROOT}"
print("PROJ_ROOT:", PROJ_ROOT)

## 3. Imports from `src/` — nothing is reimplemented

Bootstrap generation, group-aware CV, the fitness function, the knee reduction and the
stability metrics are **imported**. Writing parallel versions is how the duplicate-sample
leakage would silently come back.


In [ ]:
sys.path.insert(0, str(SRC_DIR))

import numpy as np, json, time
import data as dm
import nested_cv as ncv
import ga as ga_mod
import bootstrap as bs
import stability as stab
from stability_ci import pairwise_jaccard, nogueira_stability

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

print("imported:", [m.__name__ for m in (dm, ncv, ga_mod, bs, stab)])
print("knee reduction ->", stab.knee_point_mask.__doc__.strip().splitlines()[0])

## 4. Measure the NSGA-II budget

Instruments the real `eval_fn`. Reports the spread across seeds; the matched Random Search
budget is the mean rounded up, per dataset.


In [ ]:
def measure_budget(ds, seeds):
    X, y, tag = dm.load(ds); p = X.shape[1]
    counts, exposure = [], []
    for s in seeds:
        rng = np.random.default_rng(0)
        bidx = bs.stratified_bootstrap_indices(y, rng)      # first draw
        Xb, yb = X[bidx], y[bidx]
        cv = ncv.make_cv(len(yb), seed=0, groups=bidx)
        st = {"n": 0, "seen": np.zeros(p, bool)}
        def ef(mask, Xb=Xb, yb=yb, cv=cv, g=bidx, st=st):
            st["n"] += 1; st["seen"] |= mask
            return ncv.eval_mask(Xb, yb, mask, cv, groups=g)
        ga_mod.run_nsga2(p, ef, pop_size=POP_SIZE, n_gen=N_GEN, seed=s)
        counts.append(st["n"]); exposure.append(int(st["seen"].sum()))
    return counts, exposure

if BUDGET_FILE.exists():
    budget = json.load(open(BUDGET_FILE))
    print("loaded cached budget measurement")
else:
    budget = {}
    for ds in DATASETS:
        t0 = time.time()
        counts, exposure = measure_budget(ds, BUDGET_SEEDS)
        X, _, _ = dm.load(ds); p = X.shape[1]
        budget[ds] = {"counts": counts, "mean": float(np.mean(counts)),
                      "budget": int(np.ceil(np.mean(counts))),
                      "min": int(min(counts)), "max": int(max(counts)),
                      "p": p, "exposure": exposure,
                      "exposure_pct": [round(100*e/p, 1) for e in exposure]}
        print(f"{ds:>14}: evals {counts} -> budget {budget[ds]['budget']} | "
              f"exposure {budget[ds]['exposure_pct']}% [{time.time()-t0:.0f}s]", flush=True)
    json.dump(budget, open(BUDGET_FILE, "w"), indent=2)

for ds in DATASETS:
    print(f"  {ds:>14}: matched RS budget = {budget[ds]['budget']} evaluations")

## 5. Random Search

Same data, preprocessing, CV, fitness function, run count and OOB evaluation as NSGA-II.
The **only** difference is the search mechanism.


In [ ]:
def sample_mask(p, rng, k_max_cap=K_MAX_CAP):
    """Decision 3: k ~ Uniform{1..min(p, 50)}, then k features uniformly w/o replacement."""
    k_max = min(p, k_max_cap)
    k = int(rng.integers(1, k_max + 1))
    mask = np.zeros(p, dtype=bool)
    mask[rng.choice(p, k, replace=False)] = True
    return mask

def pareto_front(samples):
    """Non-dominated set w.r.t. (maximize acc, minimize n_sel).
    samples: list of (mask, acc, n_sel). Returns the same tuple format as
    ga.run_nsga2 so stability.py's reductions apply unchanged."""
    front = []
    for i, (mi, ai, ki) in enumerate(samples):
        dominated = False
        for j, (mj, aj, kj) in enumerate(samples):
            if i == j:
                continue
            if (aj >= ai and kj <= ki) and (aj > ai or kj < ki):
                dominated = True; break
        if not dominated:
            front.append((mi, ai, ki))
    # de-duplicate identical objective pairs, keep sorted by size like run_nsga2 does
    seen, out = set(), []
    for m, a, k in sorted(front, key=lambda t: t[2]):
        key = (a, k)
        if key in seen:
            continue
        seen.add(key); out.append((m, a, k))
    return out

def run_random_search(X, y, bidx, budget, seed, p):
    """One RS run on one (already drawn) bootstrap sample."""
    rng = np.random.default_rng(seed)
    Xb, yb = X[bidx], y[bidx]
    cv = ncv.make_cv(len(yb), seed=0, groups=bidx)
    samples = []
    for _ in range(budget):
        m = sample_mask(p, rng)
        acc = ncv.eval_mask(Xb, yb, m, cv, groups=bidx)
        samples.append((m, acc, int(m.sum())))
    return pareto_front(samples), len(samples)

## 6. Out-of-bag evaluation

Mirrors `results/oob_full.py`. Cell 8 asserts it reproduces the published NSGA-II OOB
values, which validates this path before it is used on Random Search.


In [ ]:
def balanced_acc(yt, yp):
    cs = np.unique(yt)
    if len(cs) < 2:
        return float((yp == yt).mean())
    return float(np.mean([(yp[yt == c] == c).mean() for c in cs]))

def oob_accuracy(X, y, bidx, features):
    """Train on in-bag rows restricted to `features`, evaluate on out-of-bag samples."""
    oob = np.setdiff1d(np.arange(len(y)), np.unique(bidx))
    if len(oob) < 5 or len(np.unique(y[oob])) < 2:
        return None, len(oob)
    Xtr, ytr = X[np.ix_(bidx, features)], y[bidx]
    Xte, yte = X[np.ix_(oob, features)], y[oob]
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    clf = KNeighborsClassifier(n_neighbors=min(5, len(ytr))); clf.fit(Xtr, ytr)
    return balanced_acc(yte, clf.predict(Xte)), len(oob)

## 7. Run — checkpointed after each (dataset, condition)

In [ ]:
results = json.load(open(OUT_FILE)) if OUT_FILE.exists() else {}

for ds in DATASETS:
    X, y, tag = dm.load(ds); p = X.shape[1]
    B = budget[ds]["budget"]

    for cond in ["bootstrap", "fixed"]:
        key = f"{ds}|{cond}"
        if key in results:
            print(f"SKIP {key} (done)"); continue

        # Reproduce EXACTLY the bootstrap draws used by the saved NSGA-II runs.
        rng_boot = np.random.default_rng(0)
        draws = [bs.stratified_bootstrap_indices(y, rng_boot) for _ in range(N_RUNS)]
        if cond == "fixed":
            draws = [draws[0]] * N_RUNS          # decision: same single draw every run

        runs, t0 = [], time.time()
        for i in range(N_RUNS):
            seed = RS_SEED_BASE[cond] + i
            bidx = draws[i]
            front, n_evals = run_random_search(X, y, bidx, B, seed, p)
            knee = stab.knee_point_mask(front)
            feats = np.where(knee)[0].tolist() if knee is not None else []
            oob_acc, oob_n = oob_accuracy(X, y, bidx, feats) if feats else (None, 0)
            runs.append({
                "dataset": ds, "method": "random_search", "condition": cond,
                "seed": int(seed), "bootstrap_id": i,
                "n_evals": int(n_evals), "front_size": len(front),
                "knee_features": feats, "knee_n_features": len(feats),
                "knee_fitness": float([a for m, a, k in front
                                       if np.array_equal(m, knee)][0]) if feats else None,
                "oob_accuracy": oob_acc, "oob_set_size": int(oob_n),
                "front": [{"n_sel": int(k), "acc": float(a),
                           "features": np.where(m)[0].tolist()} for m, a, k in front],
            })
            print(f"  [{ds}/{cond}] run {i+1}/{N_RUNS} front={len(front)} "
                  f"k={len(feats)} oob={oob_acc if oob_acc is None else round(oob_acc,3)} "
                  f"[{time.time()-t0:.0f}s]", flush=True)

        results[key] = {"budget": B, "n_runs": N_RUNS, "p": p, "runs": runs}
        json.dump(results, open(OUT_FILE, "w"), indent=2)
        print(f"--- CHECKPOINT {key} saved ---", flush=True)

print("ALL DONE ->", OUT_FILE)

## 8. Sanity checks — assertions that stop the notebook if violated

In [ ]:
import itertools

# A. OOB path reproduces the PUBLISHED NSGA-II values (validates cell 6 before we trust it)
published = {"breast_cancer": 0.930, "colon": 0.680, "leukemia": 0.774}
nsga_raw = json.load(open(RESULTS_DIR / "final_raw_results.json"))
bc_raw   = json.load(open(RESULTS_DIR / "breast_cancer_matched_result.json"))
for ds in DATASETS:
    X, y, _ = dm.load(ds)
    runs = (bc_raw if ds == "breast_cancer" else nsga_raw[ds])["bootstrap_fixed"]["raw_runs"]
    rng_b = np.random.default_rng(0); accs = []
    for r in runs:
        bidx = bs.stratified_bootstrap_indices(y, rng_b)
        front = [s for s in (r["front"] if isinstance(r, dict) else r) if s["n_sel"] > 0]
        if not front: continue
        tup = [(np.isin(np.arange(X.shape[1]), s["features"]), s["acc"], s["n_sel"]) for s in front]
        knee = stab.knee_point_mask(tup)
        a, _ = oob_accuracy(X, y, bidx, np.where(knee)[0].tolist())
        if a is not None: accs.append(a)
    got = float(np.median(accs))
    assert abs(got - published[ds]) < 0.002, f"OOB path mismatch {ds}: {got} vs {published[ds]}"
    print(f"  OK  OOB path reproduces {ds}: {got:.3f}")

# B. Budget actually matched
for ds in DATASETS:
    for cond in ["bootstrap", "fixed"]:
        for r in results[f"{ds}|{cond}"]["runs"]:
            assert r["n_evals"] == budget[ds]["budget"], f"budget mismatch {ds}/{cond}"
print("  OK  every RS run used the measured NSGA-II budget")

# C. Cardinality distribution is the pre-registered one
for ds in DATASETS:
    p = budget[ds]["p"]; kmax = min(p, K_MAX_CAP)
    ks = [s["n_sel"] for c in ["bootstrap","fixed"] for r in results[f"{ds}|{c}"]["runs"] for s in r["front"]]
    assert min(ks) >= 1 and max(ks) <= kmax, f"cardinality out of range {ds}: {min(ks)}-{max(ks)}"
print("  OK  all sampled cardinalities within 1..min(p,50)")

# D. No OOB sample leaked into its own training bootstrap
for ds in DATASETS:
    X, y, _ = dm.load(ds)
    rng_b = np.random.default_rng(0)
    for i in range(N_RUNS):
        bidx = bs.stratified_bootstrap_indices(y, rng_b)
        oob = np.setdiff1d(np.arange(len(y)), np.unique(bidx))
        assert len(np.intersect1d(oob, np.unique(bidx))) == 0
print("  OK  out-of-bag sets are disjoint from their training bootstraps")

# E. Feature indices valid
for ds in DATASETS:
    p = budget[ds]["p"]
    for cond in ["bootstrap","fixed"]:
        for r in results[f"{ds}|{cond}"]["runs"]:
            for s in r["front"]:
                assert all(0 <= f < p for f in s["features"])
                assert len(set(s["features"])) == s["n_sel"]
print("  OK  feature indices valid and duplicate-free")

# F. Reproducibility: same seed -> same output
ds = "breast_cancer"; X, y, _ = dm.load(ds); p = X.shape[1]
rng_b = np.random.default_rng(0); bidx0 = bs.stratified_bootstrap_indices(y, rng_b)
f1, _ = run_random_search(X, y, bidx0, 50, 12345, p)
f2, _ = run_random_search(X, y, bidx0, 50, 12345, p)
assert [(a, k) for m, a, k in f1] == [(a, k) for m, a, k in f2]
print("  OK  reruns with the same seed are identical")

print("\nALL SANITY CHECKS PASSED")

## 9. What to send back

Send **`results/phase1_random_search.json`** and **`results/phase1_measured_budget.json`**,
plus the console output of cells 4, 7 and 8.

Do not edit any pre-registered decision to make a result look better. If a sanity check
fails, send the error rather than working around it.
